# InceptentionNet Baseline (Notebook)

This notebook mirrors the repository implementation for the baseline paper:
- Data prep (MB vs non-MB)
- Preprocessing + augmentation
- InceptentionNet model
- 5-fold cross-validation training
- Evaluation against paper-reported metrics

In [1]:
from __future__ import annotations

import hashlib
import json
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import yaml
from PIL import Image, ImageFilter, ImageOps
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

## Configuration

In [2]:
CONFIG = {
    "seed": 42,
    "data": {
        "data_root": "data",
        "mb_class_name": "Meduloblastoma",
        "mb_target_count": 106,
        "non_mb_target_count": 630,
        "deduplicate": True,
        "image_size": 224,
        "gaussian_sigma": 2.0,
        "augmentation_factor": 4,
    },
    "model": {
        "stem_channels": 64,
        "branch_channels": 64,
        "attention_heads": 4,
        "dropout": 0.3,
    },
    "training": {
        "num_folds": 5,
        "batch_size": 8,
        "learning_rate": 0.005,
        "max_epochs": 40,
        "early_stopping_patience": 10,
        "lr_decay_factor": 0.5,
        "lr_decay_patience": 3,
        "num_workers": 4,
        "use_cuda": True,
    },
    "output": {
        "run_dir": "experiments/runs/inceptentionnet_baseline_notebook",
    },
}

PAPER_TARGETS = {
    "accuracy": 0.9807,
    "precision": 0.9143,
    "recall": 0.9603,
    "f1": 0.9354,
    "auc": 0.9941,
}

## Utilities

In [3]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() and CONFIG["training"]["use_cuda"] else "cpu")
print("Using device:", DEVICE)

Using device: cuda


## Transforms

In [4]:
@dataclass
class BaselineTransformConfig:
    image_size: int = 224
    gaussian_sigma: float = 2.0
    mean: tuple[float, float, float] = (0.5, 0.5, 0.5)
    std: tuple[float, float, float] = (0.5, 0.5, 0.5)


class GaussianAndEqualize:
    def __init__(self, sigma: float = 2.0) -> None:
        self.sigma = sigma

    def __call__(self, image):
        image = image.filter(ImageFilter.GaussianBlur(radius=self.sigma))
        return ImageOps.equalize(image)


class QuadAugmentDatasetTransform:
    def __init__(self, image_size: int, mean: tuple[float, float, float], std: tuple[float, float, float]) -> None:
        self.base = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            GaussianAndEqualize(sigma=2.0),
        ])
        self.aug = transforms.Compose([
            transforms.RandomRotation(degrees=(0, 45)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomAffine(degrees=0, scale=(0.8, 1.2)),
        ])
        self.to_tensor = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

    def __call__(self, image, variant_index: int) -> torch.Tensor:
        image = self.base(image)
        if variant_index > 0:
            image = self.aug(image)
        return self.to_tensor(image)


def build_eval_transform(config: BaselineTransformConfig) -> transforms.Compose:
    return transforms.Compose([
        transforms.Resize((config.image_size, config.image_size)),
        GaussianAndEqualize(sigma=config.gaussian_sigma),
        transforms.ToTensor(),
        transforms.Normalize(mean=config.mean, std=config.std),
    ])

## Dataset

In [5]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}


@dataclass
class Sample:
    path: str
    label: int


def _list_images(folder: Path) -> list[Path]:
    return [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]


def _file_md5(path: Path) -> str:
    hasher = hashlib.md5()
    with path.open("rb") as file:
        while True:
            chunk = file.read(8192)
            if not chunk:
                break
            hasher.update(chunk)
    return hasher.hexdigest()


def _deduplicate_by_hash(paths: list[Path]) -> list[Path]:
    unique: list[Path] = []
    seen: set[str] = set()
    for path in paths:
        digest = _file_md5(path)
        if digest in seen:
            continue
        seen.add(digest)
        unique.append(path)
    return unique


def build_binary_samples(
    data_root: str,
    mb_class_name: str = "Meduloblastoma",
    mb_target_count: int = 106,
    non_mb_target_count: int = 630,
    seed: int = 42,
    deduplicate: bool = True,
) -> list[Sample]:
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(f"Data root does not exist: {root}")

    class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
    mb_dir = root / mb_class_name
    if not mb_dir.exists():
        available = ", ".join(d.name for d in class_dirs)
        raise ValueError(f"MB class '{mb_class_name}' not found. Available classes: {available}")

    mb_images = sorted(_list_images(mb_dir))
    non_mb_images: list[Path] = []
    for class_dir in class_dirs:
        if class_dir.name == mb_class_name:
            continue
        non_mb_images.extend(_list_images(class_dir))

    if deduplicate:
        mb_images = _deduplicate_by_hash(mb_images)
        non_mb_images = _deduplicate_by_hash(non_mb_images)

    rng = random.Random(seed)
    rng.shuffle(mb_images)
    rng.shuffle(non_mb_images)

    if len(mb_images) < mb_target_count:
        raise ValueError(f"Not enough MB images after filtering. Required={mb_target_count}, available={len(mb_images)}")
    if len(non_mb_images) < non_mb_target_count:
        raise ValueError(f"Not enough non-MB images after filtering. Required={non_mb_target_count}, available={len(non_mb_images)}")

    selected_mb = mb_images[:mb_target_count]
    selected_non_mb = non_mb_images[:non_mb_target_count]

    samples = [Sample(path=str(path), label=1) for path in selected_mb]
    samples.extend(Sample(path=str(path), label=0) for path in selected_non_mb)
    rng.shuffle(samples)
    return samples


class FoldTrainDataset(Dataset):
    def __init__(self, samples: list[Sample], transform, augmentation_factor: int = 4) -> None:
        self.samples = samples
        self.transform = transform
        self.augmentation_factor = augmentation_factor

    def __len__(self) -> int:
        return len(self.samples) * self.augmentation_factor

    def __getitem__(self, index: int):
        base_index = index % len(self.samples)
        variant_index = index // len(self.samples)
        sample = self.samples[base_index]
        image = Image.open(sample.path).convert("RGB")
        tensor = self.transform(image, variant_index=variant_index)
        return tensor, np.float32(sample.label)


class FoldEvalDataset(Dataset):
    def __init__(self, samples: list[Sample], transform) -> None:
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        sample = self.samples[index]
        image = Image.open(sample.path).convert("RGB")
        tensor = self.transform(image)
        return tensor, np.float32(sample.label)

## InceptentionNet Model

In [6]:
class ConvBnRelu(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, padding: int = 0) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ModifiedInceptionBlock(nn.Module):
    def __init__(self, in_channels: int, branch_channels: int = 64) -> None:
        super().__init__()
        self.branch_1x1 = ConvBnRelu(in_channels, branch_channels, kernel_size=1)
        self.branch_3x3 = ConvBnRelu(in_channels, branch_channels, kernel_size=3, padding=1)
        self.branch_5x5 = ConvBnRelu(in_channels, branch_channels, kernel_size=5, padding=2)
        self.branch_downsample = ConvBnRelu(in_channels, branch_channels, kernel_size=3, stride=2, padding=1)
        self.downsample_after_concat = nn.Sequential(
            nn.Conv2d(branch_channels * 4, branch_channels * 4, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(branch_channels * 4),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b1 = self.branch_1x1(x)
        b2 = self.branch_3x3(x)
        b3 = self.branch_5x5(x)
        b4 = self.branch_downsample(x)

        target_h = min(b1.shape[-2], b2.shape[-2], b3.shape[-2], b4.shape[-2])
        target_w = min(b1.shape[-1], b2.shape[-1], b3.shape[-1], b4.shape[-1])
        if b1.shape[-2:] != (target_h, target_w):
            b1 = b1[:, :, :target_h, :target_w]
        if b2.shape[-2:] != (target_h, target_w):
            b2 = b2[:, :, :target_h, :target_w]
        if b3.shape[-2:] != (target_h, target_w):
            b3 = b3[:, :, :target_h, :target_w]
        if b4.shape[-2:] != (target_h, target_w):
            b4 = b4[:, :, :target_h, :target_w]

        merged = torch.cat([b1, b2, b3, b4], dim=1)
        return self.downsample_after_concat(merged)


class SelfAttention2D(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int) -> None:
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, channels, height, width = x.shape
        tokens = x.view(batch, channels, height * width).transpose(1, 2)
        attn_out, _ = self.attention(tokens, tokens, tokens)
        out = self.norm(tokens + attn_out)
        return out.transpose(1, 2).view(batch, channels, height, width)


class InceptentionNet(nn.Module):
    def __init__(self, stem_channels: int = 64, branch_channels: int = 64, num_heads: int = 4, dropout: float = 0.3) -> None:
        super().__init__()
        self.stem = ConvBnRelu(3, stem_channels, kernel_size=3, stride=1, padding=1)
        self.inception = ModifiedInceptionBlock(stem_channels, branch_channels=branch_channels)
        embed_dim = branch_channels * 4
        self.attention = SelfAttention2D(embed_dim=embed_dim, num_heads=num_heads)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(embed_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.inception(x)
        x = self.attention(x)
        x = self.pool(x)
        logits = self.classifier(x)
        return logits.squeeze(1)

## Training and Evaluation Functions

In [ ]:
@dataclass
class EarlyStopState:
    best_loss: float
    best_epoch: int
    wait: int
    best_state_dict: dict | None


def build_dataloaders(train_samples, val_samples, config: dict):
    transform_cfg = BaselineTransformConfig(
        image_size=config["data"]["image_size"],
        gaussian_sigma=config["data"]["gaussian_sigma"],
    )
    train_transform = QuadAugmentDatasetTransform(
        image_size=transform_cfg.image_size,
        mean=transform_cfg.mean,
        std=transform_cfg.std,
    )
    eval_transform = build_eval_transform(transform_cfg)

    train_dataset = FoldTrainDataset(
        train_samples,
        transform=train_transform,
        augmentation_factor=config["data"]["augmentation_factor"],
    )
    val_dataset = FoldEvalDataset(val_samples, transform=eval_transform)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["training"]["batch_size"],
        shuffle=True,
        num_workers=config["training"]["num_workers"],
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config["training"]["batch_size"],
        shuffle=False,
        num_workers=config["training"]["num_workers"],
        pin_memory=True,
    )
    return train_loader, val_loader


def evaluate(model: nn.Module, loader: DataLoader, device: torch.device, criterion: nn.Module):
    model.eval()
    losses = []
    targets: list[float] = []
    probabilities: list[float] = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            probs = torch.sigmoid(logits)
            losses.append(loss.item())
            targets.extend(labels.cpu().tolist())
            probabilities.extend(probs.cpu().tolist())

    predicted = [1 if p >= 0.5 else 0 for p in probabilities]
    targets_np = np.array(targets)
    probs_np = np.array(probabilities)
    metrics = {
        "loss": float(np.mean(losses) if losses else 0.0),
        "accuracy": float(accuracy_score(targets_np, predicted)),
        "precision": float(precision_score(targets_np, predicted, zero_division=0)),
        "recall": float(recall_score(targets_np, predicted, zero_division=0)),
        "f1": float(f1_score(targets_np, predicted, zero_division=0)),
        "auc": float(roc_auc_score(targets_np, probs_np)) if len(np.unique(targets_np)) == 2 else 0.0,
    }
    return metrics


def train_one_fold(fold_index: int, train_samples, val_samples, config: dict, device: torch.device):
    model = InceptentionNet(
        stem_channels=config["model"]["stem_channels"],
        branch_channels=config["model"]["branch_channels"],
        num_heads=config["model"]["attention_heads"],
        dropout=config["model"]["dropout"],
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["training"]["learning_rate"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=config["training"]["lr_decay_factor"],
        patience=config["training"]["lr_decay_patience"],
    )
    criterion = nn.BCEWithLogitsLoss()

    train_loader, val_loader = build_dataloaders(train_samples, val_samples, config)

    early_stop = EarlyStopState(best_loss=float("inf"), best_epoch=-1, wait=0, best_state_dict=None)
    history = []

    for epoch in range(config["training"]["max_epochs"]):
        model.train()
        epoch_losses = []
        progress = tqdm(train_loader, desc=f"Fold {fold_index + 1} | Epoch {epoch + 1}", leave=True)
        for images, labels in progress:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.4f}")

        val_metrics = evaluate(model, val_loader, device, criterion)
        train_loss = float(np.mean(epoch_losses) if epoch_losses else 0.0)
        scheduler.step(val_metrics["loss"])

        epoch_record = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            **val_metrics,
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
        }
        history.append(epoch_record)

        if val_metrics["loss"] < early_stop.best_loss:
            early_stop.best_loss = val_metrics["loss"]
            early_stop.best_epoch = epoch + 1
            early_stop.wait = 0
            early_stop.best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            early_stop.wait += 1

        if early_stop.wait >= config["training"]["early_stopping_patience"]:
            break

    if early_stop.best_state_dict is not None:
        model.load_state_dict(early_stop.best_state_dict)

    final_metrics = evaluate(model, val_loader, device, criterion)
    fold_result = {
        "fold": fold_index + 1,
        "best_epoch": early_stop.best_epoch,
        "best_val_loss": early_stop.best_loss,
        "metrics": final_metrics,
        "history": history,
    }
    return model, fold_result


def summarize_results(fold_results: list[dict]) -> dict:
    keys = ["accuracy", "precision", "recall", "f1", "auc"]
    summary = {}
    for key in keys:
        values = np.array([fold["metrics"][key] for fold in fold_results], dtype=np.float64)
        summary[key] = {
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
        }
    return summary

## Run 5-Fold Training

Uncomment and run this cell after dependencies finish installing.

In [ ]:
# samples = build_binary_samples(
#     data_root=CONFIG["data"]["data_root"],
#     mb_class_name=CONFIG["data"]["mb_class_name"],
#     mb_target_count=CONFIG["data"]["mb_target_count"],
#     non_mb_target_count=CONFIG["data"]["non_mb_target_count"],
#     seed=CONFIG["seed"],
#     deduplicate=CONFIG["data"]["deduplicate"],
# )
# labels = np.array([sample.label for sample in samples])
# splitter = StratifiedKFold(n_splits=CONFIG["training"]["num_folds"], shuffle=True, random_state=CONFIG["seed"])
#
# run_dir = Path(CONFIG["output"]["run_dir"])
# run_dir.mkdir(parents=True, exist_ok=True)
#
# fold_results = []
# for fold_index, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(samples)), labels)):
#     train_samples = [samples[i] for i in train_idx]
#     val_samples = [samples[i] for i in val_idx]
#     model, fold_result = train_one_fold(fold_index, train_samples, val_samples, CONFIG, DEVICE)
#
#     model_path = run_dir / f"fold_{fold_index + 1}.pt"
#     torch.save(model.state_dict(), model_path)
#     fold_results.append(fold_result)
#
# summary = summarize_results(fold_results)
# payload = {
#     "config": CONFIG,
#     "device": str(DEVICE),
#     "num_samples": len(samples),
#     "num_mb": int(labels.sum()),
#     "num_non_mb": int(len(labels) - labels.sum()),
#     "fold_results": fold_results,
#     "summary": summary,
# }
#
# result_path = run_dir / "cv_results.json"
# with result_path.open("w", encoding="utf-8") as file:
#     json.dump(payload, file, indent=2)
#
# print(json.dumps(summary, indent=2))

## Compare Results to Paper

In [ ]:
# result_path = Path(CONFIG["output"]["run_dir"]) / "cv_results.json"
# if result_path.exists():
#     payload = json.loads(result_path.read_text(encoding="utf-8"))
#     summary = payload["summary"]
#     print("Cross-validation summary")
#     for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
#         mean = summary[metric]["mean"]
#         std = summary[metric]["std"]
#         target = PAPER_TARGETS[metric]
#         delta = mean - target
#         print(f"{metric:10s} mean={mean:.4f} std={std:.4f} target={target:.4f} delta={delta:+.4f}")
# else:
#     print("No cv_results.json found yet. Run training cell first.")

In [ ]:
PROJECT_ROOT = Path("/home/sadeniji/Desktop/CV/fpn-inceptentionnet")
CONFIG["data"]["data_root"] = str(PROJECT_ROOT / "data")
CONFIG["output"]["run_dir"] = str(PROJECT_ROOT / "experiments" / "runs" / "inceptentionnet_baseline_notebook")
CONFIG["training"]["num_workers"] = 0
CONFIG["training"]["batch_size"] = 2
CONFIG["data"]["image_size"] = 128

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

samples = build_binary_samples(
    data_root=CONFIG["data"]["data_root"],
    mb_class_name=CONFIG["data"]["mb_class_name"],
    mb_target_count=CONFIG["data"]["mb_target_count"],
    non_mb_target_count=CONFIG["data"]["non_mb_target_count"],
    seed=CONFIG["seed"],
    deduplicate=CONFIG["data"]["deduplicate"],
)
labels = np.array([sample.label for sample in samples])
splitter = StratifiedKFold(n_splits=CONFIG["training"]["num_folds"], shuffle=True, random_state=CONFIG["seed"])

run_dir = Path(CONFIG["output"]["run_dir"])
run_dir.mkdir(parents=True, exist_ok=True)

fold_results = []
for fold_index, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(samples)), labels)):
    train_samples = [samples[i] for i in train_idx]
    val_samples = [samples[i] for i in val_idx]
    model, fold_result = train_one_fold(fold_index, train_samples, val_samples, CONFIG, DEVICE)

    model_path = run_dir / f"fold_{fold_index + 1}.pt"
    torch.save(model.state_dict(), model_path)
    fold_results.append(fold_result)

summary = summarize_results(fold_results)
payload = {
    "config": CONFIG,
    "device": str(DEVICE),
    "num_samples": len(samples),
    "num_mb": int(labels.sum()),
    "num_non_mb": int(len(labels) - labels.sum()),
    "fold_results": fold_results,
    "summary": summary,
}

result_path = run_dir / "cv_results.json"
with result_path.open("w", encoding="utf-8") as file:
    json.dump(payload, file, indent=2)

print("Saved:", result_path)
print(json.dumps(summary, indent=2))

print("\nComparison vs paper:")
for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    mean = summary[metric]["mean"]
    std = summary[metric]["std"]
    target = PAPER_TARGETS[metric]
    delta = mean - target
    print(f"{metric:10s} mean={mean:.4f} std={std:.4f} target={target:.4f} delta={delta:+.4f}")